In [ ]:
# Notebook 08: Diabetes Prediction Using Machine Learning and Deep Learning

## MSc Data Science and Analytics

### Explainable Machine Learning versus Deep Learning for Early Disease Prediction: A Multi-Dataset Comparative Study

This notebook performs preprocessing, develops machine learning and deep learning models, evaluates predictive performance,
and applies explainable artificial intelligence techniques to the Pima Indians Diabetes dataset.

In [ ]:
## Objectives

The objectives of this notebook are to:

- Handle missing clinical measurements.
- Split the dataset into training and testing sets.
- Standardise numerical features.
- Train multiple machine learning models.
- Develop a Deep Neural Network.
- Evaluate model performance.
- Save trained models.
- Generate a comparison table.

In [1]:
from pathlib import Path

import joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes"

MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH = PROJECT_ROOT / "results"

MODELS_PATH.mkdir(exist_ok=True)

RESULTS_PATH.mkdir(exist_ok=True)

In [3]:
df = pd.read_csv(
    RAW_PATH / "diabetes.csv.csv"
)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
columns_with_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

df[columns_with_missing] = df[columns_with_missing].replace(0, np.nan)

df.isnull().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

In [5]:
imputer = SimpleImputer(
    strategy="median"
)

df[columns_with_missing] = imputer.fit_transform(
    df[columns_with_missing]
)

df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [6]:
X = df.drop(
    "Outcome",
    axis=1
)

y = df["Outcome"]

print(X.shape)
print(y.shape)

(768, 8)
(768,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(X_train.shape)
print(X_test.shape)

(614, 8)
(154, 8)


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [11]:
joblib.dump(
    scaler,
    MODELS_PATH / "diabetes_scaler.pkl"
)

joblib.dump(
    imputer,
    MODELS_PATH / "diabetes_imputer.pkl"
)

print("Preprocessing objects saved.")

Preprocessing objects saved.


In [12]:
logistic = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic.fit(
    X_train_scaled,
    y_train
)

logistic_pred = logistic.predict(X_test_scaled)

logistic_prob = logistic.predict_proba(X_test_scaled)[:,1]

In [13]:
print("Accuracy :", round(accuracy_score(y_test, logistic_pred),4))
print("Precision:", round(precision_score(y_test, logistic_pred),4))
print("Recall   :", round(recall_score(y_test, logistic_pred),4))
print("F1 Score :", round(f1_score(y_test, logistic_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, logistic_prob),4))

Accuracy : 0.7078
Precision: 0.6
Recall   : 0.5
F1 Score : 0.5455
ROC AUC  : 0.813


In [14]:
joblib.dump(
    logistic,
    MODELS_PATH / "diabetes_logistic_regression.pkl"
)

['C:\\Users\\Syed SAAD ALI\\Desktop\\healthcare-ai-dissertation\\models\\diabetes_logistic_regression.pkl']

In [15]:
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

In [16]:
print("Accuracy :", round(accuracy_score(y_test, rf_pred),4))
print("Precision:", round(precision_score(y_test, rf_pred),4))
print("Recall   :", round(recall_score(y_test, rf_pred),4))
print("F1 Score :", round(f1_score(y_test, rf_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, rf_prob),4))

Accuracy : 0.7792
Precision: 0.7273
Recall   : 0.5926
F1 Score : 0.6531
ROC AUC  : 0.8192


In [17]:
joblib.dump(
    rf,
    MODELS_PATH / "diabetes_random_forest.pkl"
)

['C:\\Users\\Syed SAAD ALI\\Desktop\\healthcare-ai-dissertation\\models\\diabetes_random_forest.pkl']

In [18]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(
    X_train,
    y_train
)

xgb_pred = xgb.predict(X_test)

xgb_prob = xgb.predict_proba(X_test)[:,1]

In [19]:
print("Accuracy :", round(accuracy_score(y_test, xgb_pred),4))
print("Precision:", round(precision_score(y_test, xgb_pred),4))
print("Recall   :", round(recall_score(y_test, xgb_pred),4))
print("F1 Score :", round(f1_score(y_test, xgb_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, xgb_prob),4))

Accuracy : 0.7597
Precision: 0.6735
Recall   : 0.6111
F1 Score : 0.6408
ROC AUC  : 0.8081


In [20]:
joblib.dump(
    xgb,
    MODELS_PATH / "diabetes_xgboost.pkl"
)

['C:\\Users\\Syed SAAD ALI\\Desktop\\healthcare-ai-dissertation\\models\\diabetes_xgboost.pkl']

In [21]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6599 - loss: 0.6166 - val_accuracy: 0.6585 - val_loss: 0.5983
Epoch 2/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6802 - loss: 0.5730 - val_accuracy: 0.6829 - val_loss: 0.5659
Epoch 3/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7108 - loss: 0.5446 - val_accuracy: 0.7073 - val_loss: 0.5460
Epoch 4/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7210 - loss: 0.5240 - val_accuracy: 0.7073 - val_loss: 0.5304
Epoch 5/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7291 - loss: 0.5090 - val_accuracy: 0.7317 - val_loss: 0.5183
Epoch 6/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7454 - loss: 0.4979 - val_accuracy: 0.7642 - val_loss: 0.5076
Epoch 7/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7576 - loss: 0.4877 - val_accuracy: 0.7561 - val_loss: 0.5007
Epoch 8/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7576 - loss: 0.4804 - val_accuracy: 0.7480 - 

In [22]:
dl_prob = model.predict(X_test_scaled)

dl_pred = (dl_prob > 0.5).astype(int)

print("Accuracy :", round(accuracy_score(y_test, dl_pred),4))
print("Precision:", round(precision_score(y_test, dl_pred),4))
print("Recall   :", round(recall_score(y_test, dl_pred),4))
print("F1 Score :", round(f1_score(y_test, dl_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, dl_prob),4))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Accuracy : 0.7468
Precision: 0.6667
Recall   : 0.5556
F1 Score : 0.6061
ROC AUC  : 0.8176


In [23]:
model.save(
    MODELS_PATH / "diabetes_deep_learning.keras"
)

print("Deep Learning model saved.")

Deep Learning model saved.


In [24]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Deep Learning"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, dl_pred)
    ],
    "Precision": [
        precision_score(y_test, logistic_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, xgb_pred),
        precision_score(y_test, dl_pred)
    ],
    "Recall": [
        recall_score(y_test, logistic_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, xgb_pred),
        recall_score(y_test, dl_pred)
    ],
    "F1 Score": [
        f1_score(y_test, logistic_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred),
        f1_score(y_test, dl_pred)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, logistic_prob),
        roc_auc_score(y_test, rf_prob),
        roc_auc_score(y_test, xgb_prob),
        roc_auc_score(y_test, dl_prob)
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.707792,0.600000,0.500000,0.545455,0.812963
1,Random Forest,0.779221,0.727273,0.592593,0.653061,0.819167
2,XGBoost,0.759740,0.673469,0.611111,0.640777,0.808148
3,Deep Learning,0.746753,0.666667,0.555556,0.606061,0.817593


In [25]:
comparison.to_csv(
    RESULTS_PATH / "diabetes_model_results.csv",
    index=False
)

print("Comparison table saved.")

Comparison table saved.


In [ ]:
# Model Performance Summary

The Logistic Regression, Random Forest, XGBoost, and Deep Learning models were successfully trained and evaluated on the Pima Indians Diabetes dataset.

Performance was assessed using Accuracy, Precision, Recall, F1-score, and ROC-AUC. These metrics provide a balanced evaluation
of each model's predictive capability, particularly in the presence of moderate class imbalance.

The best-performing model from this dataset will be used in the next notebook to perform SHAP explainability analysis,
enabling interpretation of the model's predictions and identification of the most influential clinical features associated with diabetes.

In [ ]:
# Conclusion

This notebook completed the predictive modelling phase for the Pima Indians Diabetes dataset. Following preprocessing,
four predictive models were developed and evaluated using multiple performance metrics.

The trained models and evaluation results have been saved for subsequent explainability analysis and cross-dataset comparison.
The next notebook will apply SHAP Explainability to the best-performing model in order to enhance transparency and interpretability
of the prediction process.